In [75]:
# start by loading in relevant packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plot
import joblib

pd.set_option("display.float_format", "{:,.2f}".format) # random formatting stuff

# saving the training data to training_data
training_data=pd.read_csv("/Users/annakpope/Downloads/fim500 : project/model documentation/data/cs-training.csv.zip")

In [77]:
y = training_data["SeriousDlqin2yrs"] # defining the target variable
x = training_data[["MonthlyIncome","DebtRatio","NumberOfTimes90DaysLate","RevolvingUtilizationOfUnsecuredLines","NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse"]] # defining the features

# importing the model and scaling functions
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=.20,
    random_state=42,
    stratify=y
) # splitting the training and the testing data up for this model

# cleaning the data, using the median monthly income to fill missing values
median_income = x_train["MonthlyIncome"].median()
x_train["MonthlyIncome"]=x_train["MonthlyIncome"].fillna(median_income)
x_test["MonthlyIncome"] = x_test["MonthlyIncome"].fillna(median_income)

# scaling the training and testing data sets
scaling = StandardScaler()
x_train_scaled=scaling.fit_transform(x_train)
x_test_scaled=scaling.transform(x_test)

# creating the logistic regression model
my_model=LogisticRegression()
my_model.fit(x_train_scaled,y_train) # fitting the model using the training data

# saving everything to import into the python code space
joblib.dump(my_model, "my_model.pkl")
joblib.dump(scaling, "scaling.pkl")
joblib.dump(median_income, "median_income.pkl")


['median_income.pkl']

In [74]:
# applying the model to the testing data set
pd_predictions = my_model.predict_proba(x_test_scaled)[:,1]
borrowers = np.arange(1, len(pd_predictions) + 1)

# printing out/formating the final table for the testing data set
pd_table = pd.DataFrame({
    'Borrower': borrowers,
    'PD (%)': pd_predictions*100
})


In [72]:
# creating and playing around with the function for the model created.

def predict_pd(new_data):
    
    # Select the variables used by the model
    x_new = new_data[[
        "MonthlyIncome",
        "DebtRatio",
        "NumberOfTimes90DaysLate",
        "RevolvingUtilizationOfUnsecuredLines",
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse"
    ]].copy()
    
    # Handle missing MonthlyIncome
    x_new["MonthlyIncome"] = x_new["MonthlyIncome"].fillna(median_income)
    
    # Use the scaler that was fitted on the original training data
    x_new_scaled = scaling.transform(x_new)
    
    # Predict probability of serious delinquency
    pd_predictions = my_model.predict_proba(x_new_scaled)[:, 1]
    
    # Create output table
    pd_table = pd.DataFrame({
        "Borrower": np.arange(1, len(pd_predictions) + 1),
        "PD (%)": pd_predictions*100
    })
    
    return pd_table

In [73]:
# random test to see if everything loads in properly with the function
new_data_test = pd.read_csv("/Users/annakpope/Downloads/fim500 : project/model documentation/data/cs-test.csv.zip")

new_predictions = predict_pd(new_data_test)
new_predictions

,Borrower,PD (%)
0,1,5.57
1,2,4.93
2,3,5.70
3,4,9.69
4,5,5.95
...,...,...
101498,101499,6.48
101499,101500,19.97
101500,101501,5.37
101501,101502,1.78
